#Ingesta de SBS: bronze

In [0]:
# CONFIGURACIÓN DE PARÁMETROS

dbutils.widgets.text("catalog_name", "banking_dev", "Catálogo")
dbutils.widgets.text("storage_account", "adlsprojectbank12", "Storage Account")
dbutils.widgets.dropdown("load_mode", "FULL", ["FULL"], "Modo de carga")

catalog_name = dbutils.widgets.get("catalog_name").strip()
storage_account = dbutils.widgets.get("storage_account").strip()
load_mode = dbutils.widgets.get("load_mode").strip().upper()

if not catalog_name or not storage_account:
    raise ValueError("catalog_name y storage_account son obligatorios")
if load_mode != "FULL":
    raise ValueError(f"Modo de carga no soportado: {load_mode}")

print(f"Catálogo      : {catalog_name}")
print(f"Storage       : {storage_account}")
print(f"Modo de carga : {load_mode}")

In [0]:
# CONFIGURACIÓN DE RUTAS

raw_sbs_path = f"abfss://raw@{storage_account}.dfs.core.windows.net/sbs"
bronze_base_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net"

source_paths = {
    "sbs_creditos_sector_economico": f"{raw_sbs_path}/creditos_sector_economico/sbs_creditos_sector_economico_2025-12.csv",
    "sbs_depositos_mn": f"{raw_sbs_path}/movimiento_depositos/sbs_movimiento_depositos_mn_2025-12.csv",
    "sbs_depositos_me": f"{raw_sbs_path}/movimiento_depositos/sbs_movimiento_depositos_me_2025-12.csv",
    "sbs_morosidad": f"{raw_sbs_path}/morosidad/sbs_morosidad_tipo_modalidad_2025-12_long.csv"
}
target_tables = {name: f"{catalog_name}.bronze.{name}" for name in source_paths}
target_paths = {name: f"{bronze_base_path}/{name}" for name in source_paths}

for name in source_paths:
    print(f"{name}\n  Origen : {source_paths[name]}\n  Destino: {target_tables[name]}\n  Delta  : {target_paths[name]}\n")

In [0]:
# DEFINIR ESQUEMAS

from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, LongType,
    DateType, DecimalType
)

schemas = {
    "sbs_creditos_sector_economico": StructType([
        StructField("fecha_corte", DateType(), True),
        StructField("concepto", StringType(), True),
        StructField("nivel_jerarquia", IntegerType(), True),
        StructField("numero_deudores", LongType(), True),
        StructField("creditos_moneda_nacional_miles_soles", DecimalType(18, 2), True),
        StructField("creditos_moneda_extranjera_miles_soles", DecimalType(18, 2), True),
        StructField("total_creditos_directos_miles_soles", DecimalType(18, 2), True),
        StructField("distribucion_creditos_directos_pct", DecimalType(10, 4), True)
    ]),
    "sbs_depositos_mn": StructType([
        StructField("fecha_corte", DateType(), True),
        StructField("moneda", StringType(), True),
        StructField("empresa", StringType(), True),
        StructField("numero_cuentas", LongType(), True),
        StructField("saldo_anterior_miles_soles", DecimalType(18, 2), True),
        StructField("abonos_miles_soles", DecimalType(18, 2), True),
        StructField("intereses_capitalizados_miles_soles", DecimalType(18, 2), True),
        StructField("retiros_cargos_miles_soles", DecimalType(18, 2), True),
        StructField("saldo_final_miles_soles", DecimalType(18, 2), True)
    ]),
    "sbs_depositos_me": StructType([
        StructField("fecha_corte", DateType(), True),
        StructField("moneda", StringType(), True),
        StructField("empresa", StringType(), True),
        StructField("numero_cuentas", LongType(), True),
        StructField("saldo_anterior_miles_soles", DecimalType(18, 2), True),
        StructField("abonos_miles_soles", DecimalType(18, 2), True),
        StructField("intereses_capitalizados_miles_soles", DecimalType(18, 2), True),
        StructField("retiros_cargos_miles_soles", DecimalType(18, 2), True),
        StructField("saldo_final_miles_soles", DecimalType(18, 2), True)
    ]),
    "sbs_morosidad": StructType([
        StructField("fecha_corte", DateType(), True),
        StructField("concepto_credito", StringType(), True),
        StructField("nivel_jerarquia", IntegerType(), True),
        StructField("empresa_bancaria", StringType(), True),
        StructField("morosidad_pct", DecimalType(10, 4), True)
    ])
}

print("Esquemas definidos:")
for name, schema in schemas.items():
    print(f"{name}: {len(schema.fields)} columnas")

In [0]:
# LECTURA DE ARCHIVOS CSV

dataframes = {}

for name, path in source_paths.items():
    print(f"Leyendo {name}")
    df = (
        spark.read.format("csv")
        .schema(schemas[name])
        .option("header", "true")
        .option("encoding", "UTF-8")
        .option("sep", ",")
        .option("dateFormat", "yyyy-MM-dd")
        .option("mode", "FAILFAST")
        .load(path)
    )
    dataframes[name] = df
    print(f"  Columnas: {len(df.columns)}")
    print(f"  Registros: {df.count()}\n")

print("\nLectura de los archivos completada")

In [0]:
# INCORPORACION DE INGESTION_DATE

from pyspark.sql import functions as F

dataframes = {
    name: df.withColumn("ingestion_date", F.current_timestamp())
    for name, df in dataframes.items()
}

for name, df in dataframes.items():
    if "ingestion_date" not in df.columns:
        raise ValueError(f"No se agregó ingestion_date a {name}")

    print(f"{name}: ingestion_date agregado")

In [0]:
# VALIDAR DATASETS

from pyspark.sql import functions as F

natural_keys = {
    "sbs_creditos_sector_economico": ["fecha_corte", "concepto"],
    "sbs_depositos_mn": ["fecha_corte", "empresa"],
    "sbs_depositos_me": ["fecha_corte", "empresa"],
    "sbs_morosidad": ["fecha_corte", "concepto_credito", "empresa_bancaria"]
}

quality_results = {}
for name, df in dataframes.items():
    keys = natural_keys[name]
    null_condition = None
    for key in keys:
        condition = F.col(key).isNull()
        null_condition = condition if null_condition is None else (null_condition | condition)

    null_key_count = df.filter(null_condition).count()
    duplicate_count = (
        df.groupBy(*keys).count().filter(F.col("count") > 1).count()
    )
    row_count = df.count()
    quality_results[name] = {
        "rows": row_count,
        "null_keys": null_key_count,
        "duplicate_keys": duplicate_count
    }

    print(
        f"{name:32} | Registros: {row_count:>6} "
        f"| Claves nulas: {null_key_count:>4} "
        f"| Claves duplicadas: {duplicate_count:>4}"
    )
    if row_count == 0:
        raise ValueError(f"El archivo de {name} no contiene registros.")
    if null_key_count > 0:
        raise ValueError(f"Se encontraron claves nulas en {name}.")
    if duplicate_count > 0:
        raise ValueError(f"Se encontraron claves naturales duplicadas en {name}.")

print("\nValidaciones de calidad completadas correctamente.")

In [0]:
# CARGA DE DATOS FULL A TABLAS BRONZE

load_results = {}
for name, table_name in target_tables.items():
    df_to_load = dataframes[name]
    target_columns = spark.table(table_name).columns
    df_to_load = df_to_load.select(*target_columns)
    expected_count = df_to_load.count()

    print(f"Cargando {name} en {table_name}")
    (
        df_to_load.write
        .mode("overwrite")
        .insertInto(table_name)
    )

    load_results[name] = {
        "expected": expected_count,
        "status": "LOADED"
    }
    print(f"  Registros: {expected_count}\n")

print("\nEscritura de las cuatro tablas completada.")

In [0]:
# VALIDAR CARGA

for name, table_name in target_tables.items():
    expected_count = load_results[name]["expected"]
    actual_count = spark.table(table_name).count()
    status = "SUCCESS" if expected_count == actual_count else "FAILED"

    load_results[name]["actual"] = actual_count
    load_results[name]["status"] = status

    print(f"{name:32} | Esperados: {expected_count:>6,} " f"| Obtenidos: {actual_count:>6,} | {status}")

    if status != "SUCCESS":
        raise ValueError(
            f"Error de validación en {table_name}: "
            f"esperados={expected_count}, obtenidos={actual_count}"
        )

print("\nValidación de carga finalizada")

In [0]:
# RESUMEN

total_records = sum(result["actual"] for result in load_results.values())
successful_tables = sum(
    1 for result in load_results.values()
    if result["status"] == "SUCCESS"
)

print("=" * 72)
print("RESUMEN DE INGESTA SBS - BRONZE")
print("=" * 72)
print(f"Catálogo          : {catalog_name}")
print(f"Modo de carga     : {load_mode}")
print(f"Tablas procesadas : {len(load_results)}")
print(f"Tablas exitosas   : {successful_tables}")
print(f"Total de registros: {total_records:,}")

for name, result in load_results.items():
    print(f"{name:32} | {result['actual']:>8,} | {result['status']}")

print("=" * 72)
if successful_tables == len(target_tables):
    print("RESULTADO: INGESTA SBS COMPLETADA CORRECTAMENTE")
else:
    raise RuntimeError("La ingesta SBS finalizó con errores.")
print("=" * 72)

In [0]:
from pyspark.sql import functions as F

df_morosidad = dataframes["sbs_morosidad"]

# A. Verificar duplicados incluyendo nivel_jerarquia
print("=== DUPLICADOS INCLUYENDO NIVEL JERÁRQUICO ===")

df_duplicates_level = (
    df_morosidad
    .groupBy(
        "fecha_corte",
        "concepto_credito",
        "nivel_jerarquia",
        "empresa_bancaria"
    )
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.col("count").desc())
)

print(f"Grupos duplicados: {df_duplicates_level.count()}")
display(df_duplicates_level)


# B. Revisar los registros con empresa nula
print("=== DISTRIBUCIÓN DE REGISTROS CON EMPRESA NULA ===")

display(
    df_morosidad
    .filter(F.col("empresa_bancaria").isNull())
    .groupBy("concepto_credito", "nivel_jerarquia")
    .count()
    .orderBy("concepto_credito")
)


# C. Revisar cantidad de empresas distintas
print("=== EMPRESAS IDENTIFICADAS ===")

display(
    df_morosidad
    .select("empresa_bancaria")
    .distinct()
    .orderBy("empresa_bancaria")
)

In [0]:
from pyspark.sql import functions as F

df_morosidad = dataframes["sbs_morosidad"]

# A. Revisar valores nulos por columna
print("=== VALORES NULOS POR COLUMNA ===")

df_morosidad.select([
    F.count(
        F.when(F.col(c).isNull(), 1)
    ).alias(c)
    for c in df_morosidad.columns
]).show(truncate=False)


# B. Mostrar registros que tienen alguna clave nula
print("=== REGISTROS CON CLAVES NULAS ===")

df_null_keys = df_morosidad.filter(
    F.col("fecha_corte").isNull() |
    F.col("concepto_credito").isNull() |
    F.col("empresa_bancaria").isNull()
)

display(df_null_keys)


# C. Identificar claves duplicadas
print("=== CLAVES DUPLICADAS ===")

df_duplicates = (
    df_morosidad
    .groupBy(
        "fecha_corte",
        "concepto_credito",
        "empresa_bancaria"
    )
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.col("count").desc())
)

display(df_duplicates)


# D. Mostrar algunos registros de la tabla completa
print("=== MUESTRA DE MOROSIDAD ===")

display(
    df_morosidad
    .orderBy("concepto_credito", "empresa_bancaria")
    .limit(100)
)